# ShopDesk, Section 3 Lab 1: Coordinator and Subagents

A beginner-friendly notebook that builds a **hub-and-spoke** ShopDesk: one **coordinator**
that decomposes a request, delegates each part to a focused **subagent**, and aggregates the
answers. Built on the **Claude Agent SDK**, running **Sonnet** (`claude-sonnet-4-6`) through
your **Anthropic API key**.

## The real-world scenario

A single ShopDesk agent that knows shipping rules, refund rules, and how to calm an upset
customer becomes a tangle: the rules bleed into each other and nothing is testable. The fix
is a **coordinator** that owns the big picture and hands each concern to a **subagent** with
its own prompt and tools. The catch: a subagent starts with **no memory of the parent**, so
you must pass it everything it needs, on purpose.

The question this lab answers: **how do you split a mixed request across focused subagents,
give each exactly the context it needs, and route dynamically instead of running a fixed
pipeline?**

## Objectives

- Build a **coordinator** plus **three subagents** (shipping, refund, escalation) with
  `AgentDefinition`.
- Decompose by **query type**: classify a request, then route only to the subagents it
  needs.
- Pass **structured context** (content plus metadata) explicitly, since there is no
  automatic inheritance.
- Delegate through the **Agent tool** (`allowed_tools` must include `"Agent"`).

## What you'll observe

- The pure-Python router sends a shipping-only request to one subagent and a mixed request to
  two, without a fixed pipeline.
- The hand-off packet prints a self-contained `content` plus `metadata` bundle, the only
  channel a subagent gets.
- Live, the coordinator delegates a mixed request to the right subagents and combines the
  results.

## How to run

Run top to bottom. The classifier, router, and hand-off cells are pure Python and run
anywhere. The live coordinator cell calls Claude, so paste a real key into **Setup 2/3** and
re-run from the top; otherwise it skips. **Node.js 18+** must be installed for the Agent
SDK.

## 0. Setup

**This cell:** installs the packages. The **Agent SDK** runs the coordinator and
subagents; the base SDK and dotenv come along for the key handling. The Agent SDK also needs
Node.js 18+, which cannot be pip-installed.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports what we need, pins the model, sets the `RUN_LIVE` switch, and
defines `run_async()` so the async coordinator can be called like a normal function.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # read the API key from the environment
import sys                                      # detect Windows (it needs a special event loop)
import json                                     # build and print the context payloads
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the coordinator will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **ShopDesk world**: three orders and their facts. The
coordinator and subagents reason over this, and it is the source of the metadata we pass in a
hand-off.

In [ ]:
# ===== SETUP 3/3 - the shared ShopDesk data =====
ORDERS = {                                       # our tiny order book
    "A1": {"status": 2, "refundable": True},     #   shipped,    within the window
    "A2": {"status": 3, "refundable": False},    #   delivered,  past the window
    "A3": {"status": 1, "refundable": True},     #   processing, running late
}
STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}   # status code -> human word
print("orders:", list(ORDERS))                   # quick sanity check

**This cell:** the **narrator** `stream_run()`, which runs one coordinator `query()`
and prints each delegation. Subagent calls arrive through the Agent tool, whose input carries
a `subagent_type` (which subagent) and a `prompt` (its task). We match both `"Agent"` and the
older name `"Task"` to be safe.

In [ ]:
# ===== stream one coordinator run and show its delegations =====
from claude_agent_sdk import (                     # the Agent SDK pieces we use:
    query, ClaudeAgentOptions, AgentDefinition,    #   run, options, subagent spec
    AssistantMessage, ResultMessage, TextBlock, ToolUseBlock,   # message + block types
)

async def stream_run(options, prompt):             # run query() and narrate the delegations
    print("USER:", prompt)                         #   echo the request
    answer = ""                                    #   keep the final aggregated text
    async for message in query(prompt=prompt, options=options):   # stream every message
        if isinstance(message, AssistantMessage):  #     the coordinator (or a subagent) spoke
            subs = [b for b in message.content      #       subagent invocations this turn
                    if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task")]
            if len(subs) > 1:                       #       more than one in ONE turn = parallel
                print(f"  parallel dispatch: {len(subs)} subagents in one turn")
            for b in message.content:               #       walk the blocks
                if isinstance(b, ToolUseBlock) and b.name in ("Agent", "Task"):
                    who = b.input.get("subagent_type", "?") if isinstance(b.input, dict) else "?"
                    print("  -> delegate to:", who)     # which subagent got the work
                elif isinstance(b, TextBlock):      #       coordinator text (the aggregation)
                    answer = b.text
        elif isinstance(message, ResultMessage):    #     the whole run finished
            print("  (run complete)")
    print("ANSWER:", answer)                        #   the combined answer
    return answer

### The hub-and-spoke pattern

The **coordinator** is the hub: it sees the whole request, decides how to split it, delegates
each slice, and merges the answers. Each **subagent** is a spoke: its own prompt, its own
tools, its own fresh context. Two rules make it work:

- **The Agent tool does the delegation.** Include `"Agent"` in `allowed_tools` so the
  coordinator may spawn subagents. (Some course material calls this same tool `"Task"`; the
  current SDK names it `"Agent"`.)
- **Nothing is shared unless you pass it.** A subagent starts with zero parent memory, so the
  prompt you hand it is the only context it will ever see.

---

### 🎯 Lab objective - one hub, three spokes, dynamic routing

**What you build:** a coordinator with three subagents, a pure-Python router that decomposes
by query type, and a structured hand-off packet.

**Why it helps you build real solutions:** isolating each concern keeps rules from bleeding
together and makes every part testable, and routing by need (not a fixed pipeline) is what
lets one system handle many kinds of request.

**How you'll see it:** the router picks the right subagents per request, and the coordinator
delegates a mixed request to exactly those.

**This cell:** the first two **subagents** as `AgentDefinition`s. Each has a narrow
`description` (which the coordinator reads to route), a narrow `prompt` (its role), a tool
list, and a model. Keeping shipping and refund separate is what stops their rules from
mixing.

In [ ]:
# ===== two spokes: shipping and refund =====
shipping_agent = AgentDefinition(                  # spoke 1: delivery questions only
    description="Answers delivery and tracking questions. Use for shipping status.",
    prompt="You handle shipping status only. Never discuss refunds. Be brief.",
    tools=["Read"], model="sonnet")

refund_agent = AgentDefinition(                    # spoke 2: refund policy only
    description="Decides refunds using the 30-day rule. Use for refund requests.",
    prompt="You handle refunds only. Refuse anything past the 30-day window. Be brief.",
    tools=["Read"], model="sonnet")
print("defined: shipping_agent, refund_agent")

**This cell:** the **third subagent**, for escalation. When a customer is upset, this
spoke handles the tone and flags a human. A third focused agent is cheap to add precisely
because each one is self-contained.

In [ ]:
# ===== the third spoke: escalation =====
escalation_agent = AgentDefinition(                # spoke 3: upset customers only
    description="Handles complaints and upset customers. Use when the tone is angry or urgent.",
    prompt="You de-escalate calmly, apologise once, and flag the case for a human. Be brief.",
    tools=["Read"], model="sonnet")
print("defined: escalation_agent")

**This cell:** the **coordinator** (the hub). We register all three spokes under
`agents`, give it a system prompt that says to decompose, delegate, and combine, and put
`"Agent"` in `allowed_tools` so it may actually spawn them. This object turns three spokes
into one system.

In [ ]:
# ===== the coordinator (hub) =====
COORD = ClaudeAgentOptions(                         # settings for the coordinator run
    model=MODEL,                                    #   the hub's own model
    system_prompt=("You are the ShopDesk coordinator. Split each request into parts, "
                   "delegate shipping to shipping-agent, refunds to refund-agent, and "
                   "angry or urgent cases to escalation-agent, then combine their answers."),
    agents={"shipping-agent": shipping_agent,       #   register the three spokes by name
            "refund-agent": refund_agent,
            "escalation-agent": escalation_agent},
    allowed_tools=["Agent", "Read"])                #   "Agent" unlocks delegation
print("coordinator ready with:", list(COORD.agents))

**This cell:** decomposition step one, a pure-Python **classifier**. It reads a request
and returns the set of intents present (shipping, refund, escalation) from simple signals.
This is the query-type analysis that drives dynamic routing, and it runs offline.

In [ ]:
# ===== decompose by query type: detect the intents present =====
def classify(request):                             # request text -> a set of intents
    text = request.lower()                         #   normalise for matching
    intents = set()                                #   collect what we find
    if any(w in text for w in ["where", "status", "track", "shipped", "arrive", "late", "delivery"]):
        intents.add("shipping")                    #   a shipping signal
    if any(w in text for w in ["refund", "money back", "return"]):
        intents.add("refund")                      #   a refund signal
    if any(w in text for w in ["furious", "angry", "upset", "unacceptable", "ridiculous", "complaint"]):
        intents.add("escalation")                  #   an escalation signal
    return intents or {"shipping"}                 #   default to shipping if nothing matched

for r in ["Where is A1?", "Refund A2 please", "A3 is late and I am furious"]:
    print(f"{r!r:38} ->", classify(r))             #   show the classifier offline

**This cell:** decomposition step two, the **router**, plus the fixed pipeline it
replaces. `route()` maps detected intents to subagent names, so each request goes only to the
subagents it needs. Compare it to `FIXED_PIPELINE`, which would always run every stage.

In [ ]:
# ===== route dynamically, instead of running a fixed pipeline =====
INTENT_TO_AGENT = {"shipping": "shipping-agent",   # map each intent to its spoke
                   "refund": "refund-agent",
                   "escalation": "escalation-agent"}
FIXED_PIPELINE = ["shipping-agent", "refund-agent", "escalation-agent"]   # the rigid alternative

def route(request):                                # request -> the subagents it actually needs
    return [INTENT_TO_AGENT[i] for i in sorted(classify(request))]

for r in ["Where is A1?", "Where is A1, and can I refund A2?"]:
    print(f"{r!r:42}")
    print("   dynamic:", route(r), "  vs fixed:", FIXED_PIPELINE)   # fewer, targeted calls

**This cell:** the **hand-off packet**. Because a subagent has no parent memory, we
build an explicit bundle: `content` (the task in words) and `metadata` (the order id and its
facts). That packet is the entire channel between coordinator and subagent, so it must be
self-contained.

In [ ]:
# ===== structured context passing: content + metadata =====
def handoff(request, order_id, intent):            # build a self-contained brief for one subagent
    return {
        "content": f"{intent} task: {request}",    #   WHAT to do, in words
        "metadata": {                              #   the facts the subagent cannot see otherwise
            "order_id": order_id,                  #     which order (never assume)
            "facts": ORDERS[order_id],             #     its status and refundability
            "status_word": STATUS_NAMES[ORDERS[order_id]["status"]],   # a readable status
            "intent": intent,                      #     the slice this subagent owns
        },
    }

print(json.dumps(handoff("Can I refund A2?", "A2", "refund"), indent=2))   # everything, in one packet

**This cell:** runs the **coordinator** on a mixed request that holds both a shipping
and a refund question. Watch it delegate each part to the right subagent and then combine the
results into one answer.

In [ ]:
# ===== run the coordinator on a mixed request =====
mixed = "Where is order A1 right now, and can I refund order A2?"   # two concerns, one message
if RUN_LIVE:                                      # needs a real key (and Node.js 18+)
    run_async(lambda: stream_run(COORD, mixed))   #   coordinator delegates to both spokes
else:
    print("[skipped] expected: delegate to shipping-agent for A1 and refund-agent for A2,")
    print("          then a combined reply. The router above shows exactly which two.")

| anti-pattern | what to do instead |
|---|---|
| one mega-agent that knows every rule | split concerns across focused subagents |
| forget `"Agent"` in `allowed_tools` | include it, or delegation is silently blocked |
| assume a subagent remembers the parent | pass a self-contained content + metadata packet |
| run every stage on every request | route by detected intent, not a fixed pipeline |

**Lesson:** the coordinator is the only agent that sees the whole picture. Give each
subagent a **sharp description** (so routing is honest), a **structured hand-off** (so it has
the facts), and **isolated tools** (so concerns never bleed). Decompose by query type and
route to only what is needed, rather than forcing every request through a fixed pipeline.

---

## Recap - hub, spokes, and routing

| Piece | In this lab | Course topic |
|---|---|---|
| Coordinator | the hub that splits and combines | hub-and-spoke architecture |
| Subagents | shipping, refund, escalation via `AgentDefinition` | delegation to focused agents |
| Agent tool | `allowed_tools=["Agent", ...]` | spawning subagents (older name: Task) |
| Hand-off | `content` + `metadata` packet | explicit context passing |
| Router | `classify` then `route` | decomposition by query type |

One principle to carry forward: **the coordinator decomposes and delegates; each subagent
gets only its slice and only the context you pass it.** To run live, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: send an angry, late-delivery message and
watch escalation-agent get involved. Next lab: dispatching several subagents in parallel and
comparing decomposition strategies.